In [ ]:
from os.path import sep

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("vendas.csv", sep=";")

In [ ]:
# Exibir informações gerais sobre o DataFrame
print("--- Informações do DataFrame ---")
display(df.info())

print("\n--- Resumo Estatístico de Variáveis Numéricas ---")
display(df.describe())

print("\n--- Primeiras 5 linhas do DataFrame ---")
display(df.head())

In [ ]:
# 1. Quantidade de registros e colunas
linhas, colunas = df.shape
print(f"Quantidade de registros (linhas): {linhas}")
print(f"Quantidade de variáveis (colunas): {colunas}")

# 2. Verificação de valores nulos
print("\n--- Valores Nulos por Coluna ---")
nulos = df.isnull().sum()
display(nulos[nulos > 0] if nulos.sum() > 0 else "Nenhum valor nulo encontrado!")

# 3. Anomalias numéricas encontradas (valores negativos em quantidade e total)
print("\n--- Inconsistências de Valores Negativos ---")
print(f"Registros com quantidade negativa: {len(df[df['quantidade'] < 0])}")
print(f"Registros com valor total negativo: {len(df[df['valor_total'] < 0])}")

In [ ]:
# Criando uma cópia para preservar os dados originais
df_clean = df.copy()

# 1. Removendo linhas com valores nulos na coluna 'cidade_cliente'
df_clean = df_clean.dropna(subset=['cidade_cliente'])

# 2. Filtrando inconsistências lógicas (valores maiores que zero)
df_clean = df_clean[(df_clean['quantidade'] > 0) & (df_clean['valor_total'] > 0)]

# 3. Verificando e removendo registros duplicados
duplicados_antes = df_clean.duplicated().sum()
df_clean = df_clean.drop_duplicates()

# 4. Convertendo a coluna 'data_venda' para datetime (coagindo erros em NaT e removendo-os)
df_clean['data_venda'] = pd.to_datetime(df_clean['data_venda'], errors='coerce')
linhas_data_invalida = df_clean['data_venda'].isnull().sum()
df_clean = df_clean.dropna(subset=['data_venda'])

# Verificação final pós-limpeza
print("--- Resultados do Tratamento de Dados ---")
print(f"Linhas duplicadas removidas: {duplicados_antes}")
print(f"Linhas com datas corrompidas removidas: {linhas_data_invalida}")
print(f"Registros restantes na base tratada: {df_clean.shape[0]} (Redução de {df.shape[0] - df_clean.shape[0]} linhas no total)")
print(f"Valores nulos restantes: {df_clean.isnull().sum().sum()}")
print(f"Menor quantidade na base: {df_clean['quantidade'].min()}")
print(f"Menor valor total na base: {df_clean['valor_total'].min()}")
print(f"Tipo de dado da coluna data_venda: {df_clean['data_venda'].dtype}")

In [ ]:

print("Status de venda disponíveis:", df_clean['status_venda'].unique())

# 2. Cálculo do Faturamento Bruto (soma do valor_total de todos os itens de pedidos não cancelados/gerais)
faturamento_bruto_total = df_clean['valor_total'].sum()

# 3. Cálculo do Faturamento Líquido
# Consideramos apenas vendas com status 'Concluído' (ou equivalente) para o faturamento real líquido
vendas_concluidas = df_clean[df_clean['status_venda'] == 'Concluído']
faturamento_liquido = vendas_concluidas['valor_total'].sum()

print("\n--- Resultados Financeiros ---")
print(f"Faturamento Bruto Total (Geral): R$ {faturamento_bruto_total:,.2f}")
print(f"Faturamento Líquido (Apenas Vendas Concluídas): R$ {faturamento_liquido:,.2f}")

In [ ]:
print("Lista de produtos mais vendidos", df_clean['nome_produto'])

In [ ]:
# 1. Top 10 produtos por quantidade de transações
top_10_transacoes = df_clean['nome_produto'].value_counts().head(10).reset_index()
top_10_transacoes.columns = ['Nome do Produto', 'Frequência de Vendas']

# 2. Top 10 produtos por volume total de itens vendidos
top_10_quantidade = df_clean.groupby('nome_produto')['quantidade'].sum().sort_values(ascending=False).head(10).reset_index()
top_10_quantidade.columns = ['Nome do Produto', 'Total de Itens Vendidos']

print("--- TOP 10 PRODUTOS MAIS VENDIDOS (POR TRANSAÇÕES) ---")
display(top_10_transacoes)

print("\n--- TOP 10 PRODUTOS MAIS VENDIDOS (POR VOLUME DE ITENS) ---")
display(top_10_quantidade)

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Limpeza e tratamento
df_clean = df.copy()
df_clean = df_clean.dropna(subset=['cidade_cliente'])
df_clean = df_clean[(df_clean['quantidade'] > 0) & (df_clean['valor_total'] > 0)]
df_clean = df_clean.drop_duplicates()
df_clean['data_venda'] = pd.to_datetime(df_clean['data_venda'], errors='coerce')
df_clean = df_clean.dropna(subset=['data_venda'])

# 2. Filtrar apenas as vendas concluídas
vendas_concluidas = df_clean[df_clean['status_venda'] == 'Concluído']

# 3. Agrupar por categoria de produto
faturamento_por_categoria = (
    vendas_concluidas.groupby('categoria_produto', as_index=False)['valor_total']
    .sum()
    .sort_values(by='valor_total', ascending=False)
)
faturamento_por_categoria.columns = ['Categoria', 'Faturamento Líquido (R$)']

# 4. Gerar a visualização corrigida
sns.set_theme(style="whitegrid")
fig, ax = plt.subplots(figsize=(12, 6))

sns.barplot(
    x='Faturamento Líquido (R$)',
    y='Categoria',
    hue='Categoria',
    data=faturamento_por_categoria,
    palette='viridis',
    legend=False,
    ax=ax
)

# Garante margem à direita para os rótulos não serem cortados
max_val = faturamento_por_categoria['Faturamento Líquido (R$)'].max()
ax.set_xlim(0, max_val * 1.15)

# Adiciona os rótulos formatados em Reais (BR)
for index, value in enumerate(faturamento_por_categoria['Faturamento Líquido (R$)']):
    valor_br = f" R$ {value:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
    ax.text(value, index, valor_br, va='center', fontsize=10, fontweight='bold')

ax.set_title('Faturamento Líquido por Categoria de Produto (Vendas Concluídas)', fontsize=14, pad=15)
ax.set_xlabel('Faturamento Líquido (R$)', fontsize=12)
ax.set_ylabel('Categoria', fontsize=12)

sns.despine(left=True, bottom=True)
plt.tight_layout()
plt.show()

In [ ]:
#Avaliação de equipe

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

#Carregamento dos dados necessarios
try:

    df_temp = df_clean.copy()
except NameError:

    df_temp = pd.read_csv("vendas.csv", sep=";")
    df_temp = df_temp.dropna(subset=['cidade_cliente'])
    df_temp = df_temp[(df_temp['quantidade'] > 0) & (df_temp['valor_total'] > 0)]
    df_temp = df_temp.drop_duplicates()

# Filtrar apenas as vendas concluídas para a avaliação financeira
vendas_concluidas_temp = df_temp[df_temp['status_venda'] == 'Concluído']

# 1. Agrupar as vendas concluídas por vendedor
desempenho_vendedores = vendas_concluidas_temp.groupby('nome_vendedor').agg(
    faturamento_total=('valor_total', 'sum'),
    quantidade_pedidos=('id_pedido', 'nunique'),
    ticket_medio=('valor_total', 'mean')
).reset_index()

# 2. Identificar os 5 melhores por faturamento
top_5_vendedores = desempenho_vendedores.sort_values(by='faturamento_total', ascending=False).head(5)

# Exibir a tabela formatada
print("--- TOP 5 VENDEDORES: DESEMPENHO E TICKET MÉDIO ---")
display(top_5_vendedores.style.format({
    'faturamento_total': 'R$ {:,.2f}',
    'ticket_medio': 'R$ {:,.2f}'
}).hide(axis='index'))

# 3. Gerar gráfico comparativo de faturamento
plt.figure(figsize=(10, 5))
sns.set_theme(style="whitegrid")
ax = sns.barplot(
    x='faturamento_total',
    y='nome_vendedor',
    data=top_5_vendedores,
    hue='nome_vendedor',
    palette='Blues_r',
    legend=False
)

# Adicionar os rótulos de valor nas barras
for index, value in enumerate(top_5_vendedores['faturamento_total']):
    ax.text(value, index, f' R$ {value:,.2f}', va='center', fontsize=10, fontweight='bold')

plt.title('Top 5 Vendedores por Faturamento Líquido (Vendas Concluídas)', fontsize=14, pad=15)
plt.xlabel('Faturamento Líquido (R$)', fontsize=12)
plt.ylabel('Vendedor', fontsize=12)
plt.tight_layout()
plt.show()

